# Explainable Fake News Detection using ML & DL
## LIME & SHAP Based XAI

This notebook demonstrates a complete pipeline for detecting fake news using:
- **Machine Learning Models**: Logistic Regression, Random Forest, SVM
- **Deep Learning Models**: LSTM, BERT
- **Explainability**: LIME and SHAP for interpretable AI

---

## 1. Setup and Imports

In [ ]:
# Core imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Project modules
from src.data_preprocessing import DataPreprocessor, create_sample_dataset
from src.ml_models import FakeNewsMLModels
from src.explainability import ModelExplainer
from src.visualization import (
    set_style,
    plot_word_cloud,
    plot_text_length_distribution,
    plot_multiple_roc_curves,
    plot_multiple_confusion_matrices,
    create_performance_report
)
from src.utils import (
    create_directories,
    evaluate_model,
    plot_confusion_matrix,
    plot_roc_curve,
    compare_models,
    get_feature_importance,
    plot_feature_importance
)

# Set visualization style
set_style()
%matplotlib inline

print("✓ All imports successful!")
print(f"NumPy version: {np.__version__}")
print(f"Pandas version: {pd.__version__}")

## 2. Create Sample Dataset

For demonstration purposes, we'll create a sample fake news dataset. 
In production, you would use real datasets like:
- Kaggle Fake News Dataset
- LIAR Dataset  
- FakeNewsNet
- ISOT Fake News Dataset

In [ ]:
# Create sample dataset
create_directories()
df = create_sample_dataset('data/raw/sample_news.csv', n_samples=2000)

print(f"\nDataset shape: {df.shape}")
print(f"\nFirst few samples:")
df.head(10)

## 3. Exploratory Data Analysis

In [ ]:
# Label distribution
print("Label Distribution:")
print(df['label'].value_counts())
print(f"\nPercentage:")
print(df['label'].value_counts(normalize=True) * 100)

# Plot distribution
plt.figure(figsize=(8, 6))
df['label'].value_counts().plot(kind='bar')
plt.title('Label Distribution')
plt.xlabel('Label (0=Real, 1=Fake)')
plt.ylabel('Count')
plt.xticks([0, 1], ['Real', 'Fake'], rotation=0)
plt.show()

In [ ]:
# Text length analysis
plot_text_length_distribution(df)

In [ ]:
# Word clouds
plot_word_cloud(
    df['text'].tolist(),
    df['label'].values,
    label_value=0,
    title='Word Cloud - Real News'
)

In [ ]:
plot_word_cloud(
    df['text'].tolist(),
    df['label'].values,
    label_value=1,
    title='Word Cloud - Fake News'
)

## 4. Data Preprocessing

Steps:
1. Text cleaning (remove URLs, special characters, etc.)
2. Tokenization
3. Stopword removal
4. Lemmatization
5. TF-IDF vectorization

In [ ]:
# Initialize preprocessor
preprocessor = DataPreprocessor(max_features=5000)

# Preprocess text
df = preprocessor.preprocess_dataframe(df)

# Show example of preprocessing
print("Original Text:")
print(df['text'].iloc[0])
print("\nProcessed Text:")
print(df['processed_text'].iloc[0])

In [ ]:
# Split data
X_train, X_val, X_test, y_train, y_val, y_test = \
    preprocessor.prepare_train_test_split(df, test_size=0.2, val_size=0.1)

# Create TF-IDF features
X_train_tfidf, X_val_tfidf, X_test_tfidf = \
    preprocessor.create_tfidf_features(X_train, X_val, X_test)

## 5. Train Machine Learning Models

We'll train three traditional ML models:
- **Logistic Regression**: Fast, interpretable baseline
- **Random Forest**: Ensemble method with feature importance
- **SVM**: Effective for high-dimensional text data

In [ ]:
# Initialize ML models
ml_models = FakeNewsMLModels(random_state=42)

# Create models
ml_models.create_logistic_regression()
ml_models.create_random_forest(n_estimators=100, max_depth=20)
ml_models.create_svm()

In [ ]:
# Train Logistic Regression
ml_models.train_model('logistic_regression', X_train_tfidf, y_train, X_val_tfidf, y_val)

In [ ]:
# Train Random Forest
ml_models.train_model('random_forest', X_train_tfidf, y_train, X_val_tfidf, y_val)

In [ ]:
# Train SVM
ml_models.train_model('svm', X_train_tfidf, y_train, X_val_tfidf, y_val)

## 5.5 Deep Learning Models (LSTM & BERT)

Below we demonstrate how to create, train (briefly for demo) and save deep learning models:\
- LSTM: a bidirectional LSTM trained on tokenized sequences\
- BERT: fine-tuning a pretrained BERT using the Hugging Face Transformers library\

Note: these cells use small subsets and few epochs to keep runtime short during demos.

In [ ]:
# Deep Learning setup: imports and helper functions
from src.dl_models import FakeNewsDeepLearning
import torch
from torch.utils.data import TensorDataset, DataLoader
import numpy as np
from collections import Counter

# Instantiate DL helper (auto-selects GPU if available)
dl = FakeNewsDeepLearning()

# We'll use a small subset for the demo to keep runtime short
demo_train_size = min(1000, len(X_train))
demo_val_size = min(200, len(X_val))
X_train_dl = X_train[:demo_train_size]
y_train_dl = np.array(y_train[:demo_train_size])
X_val_dl = X_val[:demo_val_size]
y_val_dl = np.array(y_val[:demo_val_size])
X_test_dl = X_test[:200]
y_test_dl = np.array(y_test[:200])

# Build a simple vocabulary from tokenized training data using the preprocessor
def build_vocab(texts, preprocessor, max_vocab=5000):
    token_lists = [preprocessor.tokenize_and_lemmatize(t) for t in texts]
    counter = Counter([tok for tokens in token_lists for tok in tokens])
    most_common = counter.most_common(max_vocab)
    vocab = {tok: idx+1 for idx, (tok, _) in enumerate(most_common)}  # reserve 0 for padding
    return vocab

vocab = build_vocab(X_train_dl, preprocessor, max_vocab=2000)
vocab_size = len(vocab) + 1
print(f'Vocab size (demo): {vocab_size}')

# Convert texts to fixed-length sequences of token ids
def texts_to_sequences(texts, preprocessor, vocab, max_len=100):
    seqs = []
    for t in texts:
        toks = preprocessor.tokenize_and_lemmatize(t)
        ids = [vocab.get(w, 0) for w in toks][:max_len]
        # pad
        if len(ids) < max_len:
            ids = ids + [0] * (max_len - len(ids))
        seqs.append(ids)
    return np.array(seqs)

max_seq_len = min( preprocessor.max_len, 100)
X_train_seq = texts_to_sequences(X_train_dl, preprocessor, vocab, max_len=max_seq_len)
X_val_seq = texts_to_sequences(X_val_dl, preprocessor, vocab, max_len=max_seq_len)
X_test_seq = texts_to_sequences(X_test_dl, preprocessor, vocab, max_len=max_seq_len)

# Create DataLoaders for LSTM
batch_size = 32
train_dataset = TensorDataset(torch.LongTensor(X_train_seq), torch.LongTensor(y_train_dl))
val_dataset = TensorDataset(torch.LongTensor(X_val_seq), torch.LongTensor(y_val_dl))
test_dataset = TensorDataset(torch.LongTensor(X_test_seq), torch.LongTensor(y_test_dl))
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size)
test_loader = DataLoader(test_dataset, batch_size=batch_size)

print('DL DataLoaders ready (LSTM).')

In [ ]:
# Create, train and save a small LSTM model (demo run - few epochs)
lstm_model = dl.create_lstm_model(vocab_size=vocab_size, embedding_dim=64, hidden_dim=128, num_layers=1, dropout=0.3)

# Train for a very small number of epochs so demo runs quickly
try:
    history_lstm = dl.train_lstm(lstm_model, train_loader, val_loader=val_loader, epochs=2, learning_rate=0.001)
    dl.save_model('lstm', 'models/lstm.pt')
    print('LSTM training (demo) complete and model saved to models/lstm.pt')
except Exception as e:
    print('LSTM training skipped or failed:', e)

# Quick evaluation on demo test set
try:
    preds, probs = dl.predict(lstm_model, test_loader)
    acc = (preds == y_test_dl[:len(preds)]).mean()
    print(f'Demo LSTM accuracy on small test subset: {acc:.4f}')
except Exception as e:
    print('LSTM prediction failed:', e)

In [ ]:
# BERT fine-tuning (demo): use Hugging Face tokenizer and TextDataset wrapper
from transformers import BertTokenizer
from src.dl_models import TextDataset

tokenizer = BertTokenizer.from_pretrained('bert-base-uncased', do_lower_case=True)

# Create small TextDataset instances for BERT (this will tokenize inputs)
bert_train = TextDataset(list(X_train_dl), y_train_dl, tokenizer=tokenizer, max_length=128)
bert_val = TextDataset(list(X_val_dl), y_val_dl, tokenizer=tokenizer, max_length=128)
bert_test = TextDataset(list(X_test_dl), y_test_dl, tokenizer=tokenizer, max_length=128)

def collate_fn(batch):
    input_ids = torch.stack([item['input_ids'] for item in batch])
    attention_mask = torch.stack([item['attention_mask'] for item in batch])
    labels = torch.stack([item['label'] for item in batch])
    return {'input_ids': input_ids, 'attention_mask': attention_mask, 'label': labels}

bert_train_loader = DataLoader(bert_train, batch_size=8, shuffle=True, collate_fn=collate_fn)
bert_val_loader = DataLoader(bert_val, batch_size=8, collate_fn=collate_fn)
bert_test_loader = DataLoader(bert_test, batch_size=8, collate_fn=collate_fn)

# Create and fine-tune BERT (very small epochs for demo)
bert_model = dl.create_bert_model('bert-base-uncased')
try:
    history_bert = dl.train_bert(bert_model, bert_train_loader, val_loader=bert_val_loader, epochs=1, learning_rate=2e-5)
    dl.save_model('bert', 'models/bert_state.pt')
    print('BERT fine-tune (demo) complete and model saved to models/bert_state.pt')
except Exception as e:
    print('BERT fine-tuning skipped or failed:', e)

# Quick BERT prediction demo
try:
    preds, probs = dl.predict(bert_model, bert_test_loader)
    acc = (preds == y_test_dl[:len(preds)]).mean()
    print(f'Demo BERT accuracy on small test subset: {acc:.4f}')
except Exception as e:
    print('BERT prediction failed:', e)

## 6. Model Evaluation

In [ ]:
# Evaluate all models
results_list = []
roc_data = {}
cm_data = {}

for model_name in ['logistic_regression', 'random_forest', 'svm']:
    # Predictions
    y_pred = ml_models.predict(model_name, X_test_tfidf)
    y_proba = ml_models.predict_proba(model_name, X_test_tfidf)
    
    # Evaluate
    results = evaluate_model(y_test, y_pred, model_name)
    results_list.append(results)
    
    # Store for plotting
    roc_data[model_name] = (y_test, y_proba[:, 1] if y_proba.ndim == 2 else y_proba)
    cm_data[model_name] = (y_test, y_pred)

In [ ]:
# Compare models
compare_models(results_list)

In [ ]:
# Plot ROC curves
plot_multiple_roc_curves(roc_data)

In [ ]:
# Plot confusion matrices
plot_multiple_confusion_matrices(cm_data)

## 7. Feature Importance Analysis

In [ ]:
# Logistic Regression feature importance
lr_model = ml_models.get_model('logistic_regression')
lr_importance = get_feature_importance(lr_model, preprocessor.vectorizer, top_n=20)

print("Logistic Regression - Top 10 Features:")
print(lr_importance.head(10))

plot_feature_importance(lr_importance, 'Logistic Regression')

In [ ]:
# Random Forest feature importance
rf_model = ml_models.get_model('random_forest')
rf_importance = get_feature_importance(rf_model, preprocessor.vectorizer, top_n=20)

print("Random Forest - Top 10 Features:")
print(rf_importance.head(10))

plot_feature_importance(rf_importance, 'Random Forest')

## 8. LIME Explanations

LIME (Local Interpretable Model-agnostic Explanations) explains individual predictions by:
- Perturbing the input text
- Building a local linear model
- Highlighting influential words

In [ ]:
# Select best model for explanations
best_model_result = max(results_list, key=lambda x: x['f1_score'])
best_model_name = best_model_result['model_name']
best_model = ml_models.get_model(best_model_name)

print(f"Using best model for explanations: {best_model_name}")
print(f"F1-Score: {best_model_result['f1_score']:.4f}")

# Create explainer
explainer = ModelExplainer(
    model=best_model,
    vectorizer=preprocessor.vectorizer,
    class_names=['Real', 'Fake']
)

In [ ]:
# Select a test sample
sample_idx = 10
sample_text = X_test[sample_idx]
true_label = "Fake" if y_test[sample_idx] == 1 else "Real"
pred_label = "Fake" if ml_models.predict(best_model_name, X_test_tfidf[sample_idx:sample_idx+1])[0] == 1 else "Real"

print(f"Sample Text: {sample_text}")
print(f"\nTrue Label: {true_label}")
print(f"Predicted Label: {pred_label}")

In [ ]:
# Generate LIME explanation
lime_exp = explainer.explain_with_lime(sample_text, num_features=10)

# Visualize
explainer.visualize_lime_explanation(lime_exp)

In [ ]:
# Get top LIME features
lime_features = explainer.get_lime_top_features(lime_exp, top_n=15)
print("\nLIME Top Features:")
lime_features

In [ ]:
# Show LIME explanation as HTML (interactive)
lime_exp.show_in_notebook(text=True)

## 9. SHAP Explanations

SHAP (SHapley Additive exPlanations) provides:
- Theoretically grounded explanations using Shapley values from game theory
- Consistent and accurate feature attribution
- Multiple visualization types (force plots, summary plots)

In [ ]:
# Generate SHAP explanations (on a subset for speed)
shap_sample_size = 50
X_test_sample = list(X_test[:shap_sample_size])

print(f"Generating SHAP explanations for {shap_sample_size} samples...")
print("This may take a few minutes...")

shap_values, shap_explainer = explainer.explain_with_shap(
    X_test_sample,
    background_samples=30
)

In [ ]:
# SHAP summary plot
X_test_tfidf_sample = preprocessor.vectorizer.transform(X_test_sample)

explainer.visualize_shap_summary(
    shap_values,
    X_test_tfidf_sample,
    max_display=20
)

In [ ]:
# SHAP force plot for a single instance
explainer.visualize_shap_force_plot(
    shap_values,
    shap_explainer,
    instance_idx=0
)

In [ ]:
# Get top SHAP features for a sample
shap_top_features = explainer.get_shap_top_features(
    shap_values,
    instance_idx=0,
    top_n=15
)

print("\nSHAP Top Features (Sample 1):")
shap_top_features

## 10. Compare Multiple Explanations

Let's compare LIME and SHAP explanations for different samples

In [ ]:
# Explain multiple samples
for i in range(3):
    sample_idx = i * 10
    text = X_test[sample_idx]
    true_label = "Fake" if y_test[sample_idx] == 1 else "Real"
    
    print(f"\n{'='*80}")
    print(f"Sample {i+1}")
    print(f"{'='*80}")
    print(f"Text: {text[:150]}...")
    print(f"True Label: {true_label}")
    
    # LIME explanation
    lime_exp = explainer.explain_with_lime(text, num_features=8)
    lime_top = explainer.get_lime_top_features(lime_exp, top_n=5)
    
    print("\nLIME Top 5 Features:")
    print(lime_top.to_string(index=False))

## 11. Performance Summary and Conclusions

In [ ]:
# Create comprehensive performance report
report = create_performance_report(results_list)

# Create summary table
summary_df = pd.DataFrame([
    {
        'Model': r['model_name'],
        'Accuracy': f"{r['accuracy']:.4f}",
        'Precision': f"{r['precision']:.4f}",
        'Recall': f"{r['recall']:.4f}",
        'F1-Score': f"{r['f1_score']:.4f}"
    }
    for r in results_list
])

print("\n📊 Model Performance Summary:")
display(summary_df)

## Key Findings

### Model Performance
- All three ML models achieved strong performance on fake news detection
- Random Forest typically performs best due to ensemble approach
- Logistic Regression provides a fast, interpretable baseline
- SVM is effective but slower to train

### Explainability Insights

**LIME:**
- Provides local, instance-level explanations
- Easy to understand word-level contributions
- Model-agnostic - works with any classifier
- Fast generation of explanations

**SHAP:**
- Theoretically grounded in game theory
- Consistent feature attribution
- Global and local explanations
- More computationally intensive

### Important Features for Fake News Detection
Common indicators found:
- Sensationalist language
- Emotional trigger words
- Conspiracy-related terms
- Lack of specific facts and figures

---

## Next Steps

1. **Try real datasets**: Use Kaggle, LIAR, or FakeNewsNet datasets
2. **Deep Learning models**: Implement LSTM and BERT for improved performance
3. **Feature engineering**: Add metadata features (source, author, date)
4. **Ensemble methods**: Combine multiple models
5. **Deployment**: Create a web API or app for real-time detection
6. **Cross-domain testing**: Test on different types of news

---

## References

- Ribeiro, M. T., et al. (2016). "Why Should I Trust You?" Explaining the Predictions of Any Classifier. KDD.
- Lundberg, S. M., & Lee, S. I. (2017). A Unified Approach to Interpreting Model Predictions. NIPS.
- Shu, K., et al. (2017). Fake News Detection on Social Media: A Data Mining Perspective.
- Devlin, J., et al. (2018). BERT: Pre-training of Deep Bidirectional Transformers for Language Understanding.

---

**Thank you for exploring this notebook!**

For more information, check the project repository and documentation.